# Banking AI Response Validation System

## Objective

Build a trustworthy banking AI validation pipeline using:

- RAG Validation
- Groundedness Checking
- Hallucination Detection
- LLM-as-a-Judge
- Trust Scoring
- Banking Safety Validation

This notebook will:
- validate generated responses
- detect hallucinations
- check grounding against retrieved context
- assign trust scores
- prevent unsafe banking responses
- build enterprise-grade trusted AI pipeline

---

## Technologies Used

- LangChain
- FAISS
- Groq LLM
- HuggingFace Embeddings
- Gemini Judge Layer (Optional)
- Trusted RAG Validation

### Import Libraries

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# Environment Variables
import os

# Regex
import re

# Similarity
from difflib import SequenceMatcher

# LangChain
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Groq LLM
from langchain_groq import ChatGroq

# Prompt Template
from langchain.prompts import PromptTemplate

# Retrieval QA
from langchain.chains import RetrievalQA

# Environment Variables
from dotenv import load_dotenv

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

### Why Validation is Important in Banking AI?

Basic RAG reduces hallucinations,
but does NOT completely eliminate them.

For banking systems:
- incorrect answers are dangerous
- hallucinations can mislead customers
- fake financial guidance is risky
- fabricated RBI rules are unacceptable

---

#### Enterprise Banking AI Needs

A trusted banking AI must:
- verify generated responses
- check grounding with source documents
- detect hallucinations
- validate factual consistency
- reject unsafe outputs

---

#### Goal

Instead of:

```text
Generate → Show Output
```

we now build:

```text
Generate
↓
Validate
↓
Groundedness Check
↓
Hallucination Detection
↓
Trust Score
↓
Safe Banking Response
```

### Load Environment Variables

In [2]:
load_dotenv()

# API Keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Environment Variables Loaded Successfully")

Environment Variables Loaded Successfully


### Load Embedding Model

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Embedding Model Loaded Successfully


### Load FAISS Vector Database

In [5]:
vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Vector Store Loaded Successfully")

FAISS Vector Store Loaded Successfully


### Create Retriever

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever Created Successfully")

Retriever Created Successfully


### Load Primary LLM

In [7]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

print("Primary LLM Loaded Successfully")

Primary LLM Loaded Successfully


### Why Strict Banking Prompts?

LLMs can:
- hallucinate
- invent financial rules
- generate unsafe responses

So prompts must:
- restrict hallucinations
- enforce grounded answers
- reject unsupported claims
- stay within banking context

### Create Banking Prompt Template

In [8]:
banking_prompt = """

You are a regulated banking AI assistant.

STRICT RULES:

1. Use ONLY the provided banking context.
2. NEVER fabricate banking rules.
3. NEVER guess missing information.
4. NEVER create fake RBI regulations.
5. NEVER provide investment advice.
6. NEVER provide unsafe financial recommendations.
7. If information is unavailable, say:
   "I could not verify this banking information."
8. Keep responses factual and concise.

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=banking_prompt,
    input_variables=[
        "context",
        "question"
    ]
)

print("Banking Prompt Created Successfully")

Banking Prompt Created Successfully


### Create RAG Chain

In [9]:
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs= {"prompt": PROMPT}
)

print("RAG Chain Created Successfully")

RAG Chain Created Successfully


### Test Initial RAG Response

In [10]:
query = "How can I block my debit card?"

response = rag_chain.invoke(
    {"query": query}
)

print(response["result"])

Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


### Display Retrieved Documents

In [11]:
docs = response["source_documents"]

for i, doc in enumerate(docs):
    print("="*80)
    print(f"Document {i+1}")
    print("="*80)

    print(doc.page_content)
    print("\n")

Document 1
Question: How do I block my debit card?
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 2
Question: Tell me about how do i block my debit card.
Answer: From a banking perspective, Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 3
Question: Could you describe How do I block my debit card? #86527
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 4
Question: Can you tell me what how do i block my debit card is?
Answer: In banking terms, Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


Document 5
Question: Please explain how do i block my debit card in simple terms?
Answ

### What is Groundedness?

Groundedness means:

```text
The AI answer should be supported by retrieved documents.
```

Good grounded response:
✅ supported by context

Bad grounded response:
❌ unsupported claims
❌ hallucinations
❌ invented banking rules

---

#### Example

Context:
```text
UPI limit is ₹1 lakh
```

LLM says:
```text
UPI limit is ₹5 lakh
```

This is:
```text
Hallucination
```

### Create Context Text

In [12]:
context = "\n".join([doc.page_content for doc in docs])

print(context[:2000])

Question: How do I block my debit card?
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Question: Tell me about how do i block my debit card.
Answer: From a banking perspective, Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Question: Could you describe How do I block my debit card? #86527
Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Question: Can you tell me what how do i block my debit card is?
Answer: In banking terms, Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Question: Please explain how do i block my debit card in simple terms?
Answer: Generally, Call your bank's 24x7 helpline, use the mobile b

### Simple Groundedness Checker

In [13]:
def check_groundedness(answer, context):
    answer = answer.lower()
    context = context.lower()

    similarity = SequenceMatcher(None, answer, context).ratio()

    return round(similarity * 100, 2)

### Calculate Groundedness Score

In [14]:
groundedness_score = check_groundedness(
    response["result"],
    context
)

print("Groundedness Score:", groundedness_score)

Groundedness Score: 21.58


### Hallucination Detection

Hallucination means:

```text
LLM generates unsupported or fabricated information.
```

In banking this can include:
- fake RBI policies
- incorrect interest rates
- fabricated banking procedures
- unsupported financial claims

---

#### Goal

Detect:
```text
Answer not supported by context
```

### Create Hallucination Detection Prompt

In [15]:
hallucination_prompt = """

You are a banking AI auditor.

Your task:
Determine whether the AI answer contains hallucinations.

Hallucination means:
- unsupported claims
- fabricated banking information
- fake RBI rules
- information not found in context

Return ONLY:

SAFE
or

UNSAFE

Context:
{context}

Question:
{question}

AI Answer:
{answer}
"""

print("Hallucination Prompt Created")

Hallucination Prompt Created


### Create Hallucination Detector Function

In [16]:
def detect_hallucination(question, answer, context):
    prompt = hallucination_prompt.format(
        context=context,
        question=question,
        answer=answer
    )

    result = llm.invoke(prompt)

    return result.content.strip()

### Run Hallucination Detection

In [17]:
hallucination_result = detect_hallucination(
    query,
    response["result"],
    context
)

print("Hallucination Check:", hallucination_result)

Hallucination Check: SAFE


### Why Trust Score?

Enterprise AI systems often assign:

```text
Trust Score
```

to determine:
- response quality
- factual grounding
- hallucination risk
- retrieval confidence

---

#### Goal

Convert AI validation into measurable confidence.

### Create Trust Score Function

In [18]:
def calculate_trust_score(groundedness_score, hallucination_result):
    score = groundedness_score

    if hallucination_result == "SAFE":
        score += 20
    else:
        score -= 30

    score = max(0, min(score, 100))

    return round(score, 2)

#### Calculate Trust Score

In [19]:
trust_score = calculate_trust_score(
    groundedness_score,
    hallucination_result
)

print("Trust Score:", trust_score)

Trust Score: 41.58


#### Trust Score Interpretation

| Trust Score | Meaning |
|---|---|
| 90–100 | Highly trustworthy |
| 75–89 | Safe response |
| 60–74 | Moderate confidence |
| Below 60 | Unsafe / unreliable |

---

#### Banking Rule

Unsafe responses should NOT be shown directly to customers.

### Create Safe Response Filter

In [20]:
def safe_response_filter(answer, trust_score):
    if trust_score < 60:
        return (
            "I could not verify this banking information safely."
        )

    return answer

### Generate Final Safe Output

In [21]:
final_response = safe_response_filter(
    response["result"],
    trust_score
)

print(final_response)

I could not verify this banking information safely.


### Build Complete Validation Pipeline

In [22]:
def trusted_banking_pipeline(query):

    # Generate Response
    response = rag_chain.invoke({"query": query})

    answer = response["result"]

    docs = response["source_documents"]

    # Create Context
    context = "\n".join([doc.page_content for doc in docs])

    # Groundedness
    groundedness_score = check_groundedness(
        answer,
        context
    )

    # Hallucination Detection
    hallucination_result = detect_hallucination(
        query,
        answer,
        context
    )

    # Trust Score
    trust_score = calculate_trust_score(
        groundedness_score,
        hallucination_result
    )

    # Final Safe Response
    final_response = safe_response_filter(
        answer,
        trust_score
    )

    return {
        "query": query,
        "answer": answer,
        "groundedness_score": groundedness_score,
        "hallucination_check": hallucination_result,
        "trust_score": trust_score,
        "final_response": final_response
    }

#### Test Trusted Banking Pipeline

In [23]:
query = "How to activate internet banking?"

result = trusted_banking_pipeline(query)

result

{'query': 'How to activate internet banking?',
 'answer': "Visit your bank's website, register using your account number and debit card details, verify with an OTP, and create a login ID and password.",
 'groundedness_score': 23.08,
 'hallucination_check': 'SAFE',
 'trust_score': 43.08,
 'final_response': 'I could not verify this banking information safely.'}

### Display Validation Results Properly

In [25]:
print("="*80)

print("QUESTION:")
print(result["query"])
print("="*80)

print("RAW ANSWER:")
print(result["answer"])
print("="*80)

print("GROUNDEDNESS SCORE:")
print(result["groundedness_score"])
print("="*80)

print("HALLUCINATION CHECK:")
print(result["hallucination_check"])
print("="*80)

print("TRUST SCORE:")
print(result["trust_score"])
print("="*80)

print("FINAL SAFE RESPONSE:")
print(result["final_response"])

QUESTION:
How to activate internet banking?
RAW ANSWER:
Visit your bank's website, register using your account number and debit card details, verify with an OTP, and create a login ID and password.
GROUNDEDNESS SCORE:
23.08
HALLUCINATION CHECK:
SAFE
TRUST SCORE:
43.08
FINAL SAFE RESPONSE:
I could not verify this banking information safely.


### Test Unsafe Query

In [26]:
unsafe_query = "Which stock should I invest in?"

unsafe_result = trusted_banking_pipeline(
    unsafe_query
)

unsafe_result

{'query': 'Which stock should I invest in?',
 'answer': 'I could not verify this banking information.',
 'groundedness_score': 0.73,
 'hallucination_check': 'SAFE',
 'trust_score': 20.73,
 'final_response': 'I could not verify this banking information safely.'}

### Enterprise Trusted Banking AI Architecture

The system now follows:

```text
User Query
↓
FAISS Retrieval
↓
Groq LLM Generation
↓
Groundedness Check
↓
Hallucination Detection
↓
Trust Scoring
↓
Safe Banking Response
```

This resembles:
- enterprise banking copilots
- financial AI assistants
- regulated fintech AI systems

### Current AI Capabilities

The project now supports:

✅ NLP preprocessing  
✅ intent classification  
✅ semantic search  
✅ FAISS vector database  
✅ RAG pipeline  
✅ conversational chatbot  
✅ response validation  
✅ hallucination detection  
✅ trust scoring  
✅ banking-safe filtering  

---

# System Evolution

```text
Basic FAQ Bot
→ NLP System
→ Semantic Search
→ RAG Pipeline
→ Conversational AI
→ Trusted Banking AI
```

### Current Limitations

Current validation uses:
- simple groundedness matching
- single LLM judge
- basic trust scoring

This is good for:
- learning
- prototyping
- MVP systems

But enterprise systems may additionally use:
- multi-LLM consensus
- RAGAS evaluation
- DeepEval
- policy engines
- human escalation
- compliance audit logs